In [2]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [5]:
import os

In [13]:
os.listdir("../data")

['kg_test.csv', 'kg_train.csv']

In [14]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [18]:
from sklearn.model_selection import train_test_split

X = data['text']
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2)

## Data Preprocessing

In [24]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\eokushko\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [25]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [27]:
import re

In [51]:
def remove_special_chereters(text):
    return ''.join([c for c in text if c not in string.punctuation])

def remove_numbers(text):
    return re.sub(r'\d+', ' ', text)

# if surounded by white space
def remove_single_characters(text):
    return re.sub(r'\s+[a-zA-Z]\s+', ' ', text)

# white space only on the right ( from the start)
def remove_single_characters_start(text):
    return re.sub(r'\^[a-zA-Z]\s+', ' ', text)

def remove_multiple_spaces(text):
    return re.sub(r'\s+', ' ', text)

def remove_prefixed_b(text):
    return re.sub(r'^b\s+', '', text)

In [54]:
def data_cleaning(row):
    return (row
            .apply(remove_special_chereters)
            .apply(remove_numbers)
            .apply(remove_single_characters)
            .apply(remove_single_characters_start)
            .apply(remove_multiple_spaces)
            .apply(remove_prefixed_b)
            .str.lower()
        )

In [58]:
X_train = data_cleaning(X_train)
X_test = data_cleaning(X_test)

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

## Now let's work on removing stopwords
Remove the stopwords.

In [71]:
stop_words = set(stopwords.words("english"))

def stop_words_removal(row):
    return ' '.join([c for c in row.split(' ') if c not in stop_words])

In [70]:
X_train.shape

(800,)

In [74]:
X_train = X_train.apply(stop_words_removal)
X_test = X_test.apply(stop_words_removal)

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [79]:
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\eokushko\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\eokushko\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\eokushko\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [88]:
X_train_tokenized = X_train.apply(nltk.word_tokenize)

X_train_tokenized.head()

# tokenize by word

887    [good, daysirmadamplease, accept, sincere, apo...
267    [divbrdear, divdivnbspdivdivpermit, inform, de...
314    [pls, add, option, april, emilys, list, th, an...
27       [move, saebbibi, confirmed, give, enough, time]
130                                           [released]
Name: text, dtype: object

In [89]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

X_train_lemmatized = X_train_tokenized.apply(lambda words: [lemmatizer.lemmatize(w) for w in words])
print(X_train_lemmatized.head())

887    [good, daysirmadamplease, accept, sincere, apo...
267    [divbrdear, divdivnbspdivdivpermit, inform, de...
314    [pls, add, option, april, emilys, list, th, an...
27       [move, saebbibi, confirmed, give, enough, time]
130                                           [released]
Name: text, dtype: object


In [99]:
X_train = X_train_lemmatized.apply(lambda words: ' '.join(words))

In [101]:
X_test_tokenized = X_test.apply(nltk.word_tokenize)
X_test_lemmatized = X_test_tokenized.apply(lambda words: [lemmatizer.lemmatize(w) for w in words])
X_test = X_test_lemmatized.apply(lambda words: ' '.join(words))

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [114]:
X_train[y_train == 0].str.split(' ').explode().value_counts().iloc[:10]

text
pm           94
u            88
would        87
state        85
president    75
call         71
time         69
secretary    64
work         61
one          57
Name: count, dtype: int64

In [118]:
X_train[y_train == 1].str.split(' ').explode().value_counts().iloc[:10]

text
money          782
account        683
bank           632
u              582
fund           561
business       394
country        370
transaction    363
million        360
transfer       317
Name: count, dtype: int64

## Extra features

In [122]:
data_train = X_train.to_frame()
data_val = X_test.to_frame()

data_train = data_train.rename(columns={'text': 'preprocessed_text'})
data_val = data_val.rename(columns={'text': 'preprocessed_text'})

In [123]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,preprocessed_text,money_mark,suspicious_words,text_len
887,good daysirmadamplease accept sincere apologiz...,1,1,966
267,divbrdear divdivnbspdivdivpermit inform desire...,1,1,1477
314,pls add option april emilys list th anniversary,0,0,47
27,move saebbibi confirmed give enough time,0,0,40
130,released,0,0,8


## How would you create a Bag of Words with the CountVectorizer method?

In [124]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

In [127]:
X_train_bow.shape

(800, 29488)

In [128]:
X_test_bow.shape

(200, 29488)

## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [ ]:
# Your code

### Extra Task (optional) - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

Use a MultinimialNB with default parameters.

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [129]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Train
model = MultinomialNB()
model.fit(X_train_bow, y_train)

# Predict
y_pred = model.predict(X_test_bow)

# Evaluate
print(accuracy_score(y_test, y_pred))

0.955


In [130]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.93      0.96       110
           1       0.92      0.99      0.95        90

    accuracy                           0.95       200
   macro avg       0.95      0.96      0.95       200
weighted avg       0.96      0.95      0.96       200



In [131]:
# catches 99% of spam emails 
# better at catching spam than regular emails - preffered result